# Lessons 01–07
Intro, environment, RAG basics, FAQ dataset, search, prompt, and LLM setup

### Lesson 02 Environment
Setting up API keys

In [102]:
# Setup LLM Model Configuration
MODEL_ID = "gemini-3.1-flash-lite"

In [103]:
# Using OpenAI-compatible client with Gemini via Google API
from dotenv import load_dotenv
import os

# 1. Securely load hidden variables from the .env file
load_dotenv()

from openai import OpenAI

# 2. Initialize the client with Gemini configuration
openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),
    
    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 3. Make a test call to the API
openai_response = openai_client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": "Say 'Connection successful!' if you receive this."}
    ]
)

# 4. Print the output to the screen
print(openai_response.choices[0].message.content)

Connection successful!


In [73]:
# Native Gemini client setup using environment variables
import os
from dotenv import load_dotenv

# 1. Load your key into the environment
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY") 

# 2. Import the library and initialize the client
from google import genai
client = genai.Client()

# 3. Use the client to call the models
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Say 'Connection successful!' if you receive this."
)

# 4. Print the output to the screen
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Connection successful!


# Lesson 03 RAG
Search results injected into the prompt as context

In [92]:
# Using OpenAI-compatible client with Gemini Model via Google API
def openai_llm(prompt):
    openai_response = openai_client.chat.completions.create(
        model=MODEL_ID,
        messages=[
        {"role": "user", "content": prompt}
    ]
    )
    return openai_response

openai_answer = openai_llm("Hey, what's up?")
print(openai_answer.choices[0].message.content)

Hey there! Not much, just chilling. How about you? What's on your mind?


In [104]:
# Native Gemini client setup using environment variables
def llm(prompt):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response

answer = llm("Hey, what's up?")
print(answer.text)

Not much! Just hanging out in the digital void, ready to help you with whatever you need. How are things going with you? Anything on your mind today?


In [96]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer.text)

That's great you're interested! To give you the best answer, I need a little more information. Please tell me:

*   **What course are you referring to?** (e.g., the name of the course, the platform it's on, the institution offering it)
*   **Where did you discover it?** (e.g., a website, an email, a friend told you)

Once I have that information, I can help you figure out if you can still join.


In [105]:
question = "how do I cook salmon?"
answer = llm(question)
print(answer.text)

Cooking salmon is relatively simple, but because it’s a delicate fish, it’s easy to overcook. Depending on your equipment, here are the three best methods for achieving perfect, flaky salmon.

### Before you start:
*   **Pat it dry:** Always pat the salmon fillets dry with a paper towel. This helps get a better sear and prevents the fish from "steaming" in the pan.
*   **Season generously:** Salt and pepper are essential. Many people also enjoy a drizzle of olive oil, lemon juice, or garlic powder.
*   **Room Temp:** Let the salmon sit on the counter for 10–15 minutes before cooking so it cooks more evenly.

---

### Method 1: Pan-Seared (Best for Crispy Skin)
*This method gives you a golden, crunchy crust.*

1.  **Heat the pan:** Use a stainless steel or cast-iron skillet. Heat 1–2 tablespoons of oil over medium-high heat until it shimmers.
2.  **Place the fish:** Place the salmon **skin-side down**. Press down on the fillets with a spatula for 10 seconds to ensure the skin makes full

In [31]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

answer = llm(context)
print(answer.text)

Based on the information provided:

**Regarding joining the course and certificates:**

*   **Yes, you can still join the LLM Zoomcamp.**
*   **To receive a certificate, you must submit your project while submissions are still being accepted.**

**Regarding the confirmation email:**

*   **You do not need a confirmation email.** The system indicates that you are accepted, and you can begin learning and submitting homework without a formal registration or confirmation. Registration was primarily to gauge interest.

**Regarding video/Zoom links for live sessions:**

*   **Students do not receive a direct Zoom link for "Office Hours" or live/workshop sessions.**
*   **You can participate via YouTube Live.**
*   **You can submit questions for these sessions through Slido.**

**Regarding cloud alternatives with GPU:**

*   The text mentions **Google Colab, Kaggle, and Databricks** as potential cloud options with GPU capabilities.
*   It also emphasizes the importance of **checking the quota

In [34]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer.text)

Yes, you can join now. If you wish to receive a certificate, ensure you submit your project before the submission deadline.


### Lesson 04 Dataset
Course FAQ data as the RAG knowledge base

In [141]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"

# Send a request to download the JSON data from the URL
response = requests.get(docs_url)

# Convert the JSON response into usable Python data, e.g., a dict or list
courses_raw = response.json()

# Print the first 3 items in a readable JSON format
print(json.dumps(courses_raw[:3], indent=2))

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 402
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  }
]


In [120]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:

    course_url = f"""{url_prefix}{course["path"]}"""
    course_response = requests.get(course_url)

    # Stop immediately if the request failed    
    course_response.raise_for_status()

    course_data = course_response.json()

    # Add all FAQ items from this course into the main documents list
    documents.extend(course_data)

len(documents)

1342

In [121]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Lesson 05 Search
Building keyword search to retrieve the most relevant FAQ entries

In [122]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [40]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [123]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [124]:
def questions(course):
    results = index.search(
        question,
        num_results=5,
        filter_dict={"course": course}
    )
    return [doc["question"] for doc in results]

questions("mlops-zoomcamp")

['Course: How do I start?',
 'How do you remove a global data product?',
 'How do I log in to AWS ECR from the terminal using Docker?',
 'Homework 3, 2025 Cohort - Do I have to use MAGE as the orchestrator? Can I use any orchestrator I want?',
 'AWS EC2: How do I handle changing IP addresses on restart?']

In [125]:
questions("data-engineering-zoomcamp")

['Certificate: Do I need to do the homeworks to get the certificate?',
 'How do I use Git / GitHub for this course?',
 'How do I get my certificate?',
 'How do I launch Kestra?',
 'GCP: Do I need to delete my instance in Google Cloud?']

In [126]:
questions("machine-learning-zoomcamp")

['How do I submit homework?',
 'How do Lambda container images work?',
 'How do I sign up?',
 'How much time do I need for this course?',
 'Can I use LinearRegression from Scikit-Learn for this week?']

In [127]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)

In [128]:
search(question)

[{'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': '4d8c1f5b3a',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'How do I count tokens for a non-OpenAI model (Gemini, Mistral, HuggingFace)?',
  'answer': '`tiktoken` only ships tokenizers for OpenAI models. Using `cl100k_base` for other providers gives wrong counts and unreliable cost estimates.\n\nFor other providers, use their native tokenizer:\n\n- Gemini:\n  ```python\n  import google.generativeai as genai\n  model = genai.GenerativeModel(\'gemini-2.0-flash\')\n  print(model.count_tokens(prompt))\n  ```\n- Hugging Face / open-source models:\n  ```python\n  from transformers import AutoTokenizer\n  tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")\n  pr

In [132]:
question = "Certificate: Can I follow the course in a self-paced mode and get a certificate?"
search(question)

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '74eb249bbf',
  '

### Lesson 06 Building the Prompt
Building a usable prompt from search results

In [62]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [61]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [57]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [58]:
print(build_context(search_results))

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [59]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [66]:
rag_prompt = build_prompt(question, search_results)
print(rag_prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### Lesson 07 The LLM
Generating Answers from Retrieved Context

In [106]:
answer = llm(rag_prompt)
print(answer.text)

Yes, you can still join. You are welcome to start learning and submitting homework (if the forms are still open) regardless of when you joined. 

However, keep in mind that to receive a certificate, you must complete the course with the "live" cohort, which includes submitting your capstone project and performing the required peer reviews while the submission forms are open.


In [93]:
openai_llm("Hey, what's up?")

# Usage count from print(openai_llm("Hey, what's up?")) using OpenAI Python SDK
print(openai_answer.usage)

CompletionUsage(completion_tokens=20, prompt_tokens=8, total_tokens=28, completion_tokens_details=None, prompt_tokens_details=None)


In [107]:
# Access the usage metadata from llm(rag_prompt) using Gemini SDK for Python (google-genai)
print(answer.usage_metadata.prompt_token_count)      # Input tokens
print(answer.usage_metadata.candidates_token_count)  # Output tokens
print(answer.usage_metadata.total_token_count)       # Total tokens

358
75
433


In [112]:
# Updated version of the previous OpenAI-compatible Gemini function, refactored to mirror the course code
# - adds fixed instructions so the model also receives a developer message

def openai_llm(instructions, user_prompt, model=MODEL_ID):
    # Remove extra whitespace from the instruction text so it is clean before sending it to the model (good practice)
    instructions_clean = instructions.strip()

    message_history = [
        {"role": "developer", "content": instructions_clean},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )
    return response
    
openai_answer = openai_llm(INSTRUCTIONS, rag_prompt)
print(openai_answer.choices[0].message.content)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [114]:
# Updated version of the previous native Gemini client setup using environment variables
# - refactored to more closely match the course pattern
# - adds a system message so the model also receives fixed instructions

# Import Gemini SDK types so we can provide model configuration such as system_instruction separately from the user prompt
from google.genai import types 

def llm(instructions: str, user_prompt: str, model=MODEL_ID):
    # Clean the instruction text before sending it to the model
    instructions_clean = INSTRUCTIONS.strip()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_prompt,
        # Added a system message containing fixed instructions
        config=types.GenerateContentConfig(
            system_instruction=instructions_clean,
        )
    )
    return response

answer = llm(INSTRUCTIONS, rag_prompt)
print(answer.text)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [140]:
# Full RAG function, refactored to account for differences in the notebook code
# The RAG pipeline is modular, so the search backend, prompt template, or LLM can be swapped independently.

def rag(question):
    search_results = search(question) # retrieval step; e.g. swap minsearch with sqlitesearch
    user_prompt = build_prompt(question, search_results) # prompt step; e.g. swap a basic prompt for a stricter FAQ-only template with more instructions
    answer = llm(INSTRUCTIONS, user_prompt)  # generation step; can swap the SDK/provider or model (local models or free trial APIs)
    return answer

question = "Can I follow the course in a self-paced mode and get a certificate?"
answer = rag(question)
print(answer.text)

No, you cannot get a certificate if you follow the course in a self-paced mode. Certificates are only awarded if you finish the course with a "live" cohort, as the process requires you to peer-review three capstone projects, which can only be done while the course is running.


In [138]:
question = "When can I expect to receive the confirmation email?"
answer = rag(question)
print(answer.text)

You don't need a confirmation email; you are accepted. You can start learning and submitting homework without it, as registration is only used to gauge interest.


In [139]:
question = "What is the best thing about this course?"
answer = rag(question)
print(answer.text)

I don't know.
